# CIFAR-10 SNGP Checkpoint Evaluation

Load an SNGP checkpoint, inspect parameter counts, and evaluate test accuracy and NLL.

In [2]:
from pathlib import Path
import os
import sys

import torch
import yaml

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from src.data.cifar10 import get_cifar10_loaders
from src.models.sngp import SNGPResNetClassifier, laplace_predictive_probs

In [3]:
CONFIG_PATH = REPO_ROOT / "configs" / "cifar10_sngp.yaml"
CHECKPOINT_DIR = REPO_ROOT / "checkpoints_sngp"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("checkpoint dir:", CHECKPOINT_DIR)

device: cuda
checkpoint dir: /w/20252/wjcai/uq/manygp/checkpoints_sngp


In [4]:
def load_best_checkpoint(checkpoint_dir: Path, device: torch.device):
    checkpoint_paths = sorted(checkpoint_dir.glob("cifar10_sngp*.pt"))
    if not checkpoint_paths:
        raise FileNotFoundError(f"No SNGP checkpoint files found in {checkpoint_dir}")

    best_checkpoint = None
    best_path = None
    best_score = float("-inf")

    for checkpoint_path in checkpoint_paths:
        checkpoint = torch.load(checkpoint_path, map_location=device)
        score = checkpoint.get("val_accuracy")
        if score is None:
            continue
        if score > best_score:
            best_score = score
            best_checkpoint = checkpoint
            best_path = checkpoint_path

    if best_checkpoint is None or best_path is None:
        raise ValueError(f"No SNGP checkpoints with 'val_accuracy' found in {checkpoint_dir}")

    return best_path, best_checkpoint


BEST_CHECKPOINT_PATH, checkpoint = load_best_checkpoint(CHECKPOINT_DIR, device)
print("best checkpoint:", BEST_CHECKPOINT_PATH)
print("stored val accuracy:", checkpoint.get("val_accuracy"))
print("stored val nll:", checkpoint.get("val_nll"))

best checkpoint: /w/20252/wjcai/uq/manygp/checkpoints_sngp/cifar10_sngp_epoch100_acc0.9270.pt
stored val accuracy: 0.927
stored val nll: 0.35086657180786135


In [5]:
model_cfg = cfg["model"]
model = SNGPResNetClassifier(
    num_classes=model_cfg["num_classes"],
    width=model_cfg["width"],
    hidden_dim=model_cfg["hidden_dim"],
    spec_norm_bound=model_cfg["spec_norm_bound"],
    num_inducing=model_cfg["num_inducing"],
    ridge_penalty=model_cfg["ridge_penalty"],
    feature_scale=model_cfg["feature_scale"],
    gp_cov_momentum=model_cfg["gp_cov_momentum"],
    normalize_input=model_cfg["normalize_input"],
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("loaded epoch:", checkpoint.get("epoch"))

loaded epoch: 100


In [6]:
def count_parameters(module):
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    frozen = total - trainable
    return total, trainable, frozen


total_params, trainable_params, frozen_params = count_parameters(model)
backbone_total, backbone_trainable, backbone_frozen = count_parameters(model.backbone)
gp_total, gp_trainable, gp_frozen = count_parameters(model.classifier)

print(f"Model total params:     {total_params:,}")
print(f"Model trainable params: {trainable_params:,}")
print(f"Model frozen params:    {frozen_params:,}")
print()
print(f"Backbone total params:  {backbone_total:,}")
print(f"GP head total params:   {gp_total:,}")

Model total params:     11,244,746
Model trainable params: 11,244,746
Model frozen params:    0

Backbone total params:  11,234,496
GP head total params:   10,250


In [ ]:
data_cfg = cfg["data"]
_, test_loader, _, test_dataset = get_cifar10_loaders(
    data_root=data_cfg["root"],
    batch_size=data_cfg["batch_size"],
    num_workers=0,
    smoke_test=False,
)
print("test size:", len(test_dataset))

In [ ]:
@torch.no_grad()
def evaluate_sngp(model, loader, device, num_mc_samples=10):
    model.eval()
    total_correct = 0
    total_examples = 0
    total_nll = 0.0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits, variances = model(images, return_cov=True)
        probs = laplace_predictive_probs(logits, variances, num_mc_samples=num_mc_samples)
        log_probs = probs.clamp_min(1e-12).log()

        total_correct += (probs.argmax(dim=1) == labels).sum().item()
        total_examples += labels.size(0)
        total_nll += -log_probs.gather(1, labels.unsqueeze(1)).sum().item()

    return {
        "accuracy": total_correct / total_examples,
        "nll": total_nll / total_examples,
    }


metrics = evaluate_sngp(
    model=model,
    loader=test_loader,
    device=device,
    num_mc_samples=cfg["training"].get("num_mc_samples", 10),
)
print(f"Test accuracy: {metrics['accuracy'] * 100:.2f}%")
print(f"Test NLL: {metrics['nll']:.4f}")